# ESCI Classifier Training (from pre-computed embeddings)

**Before running this notebook:**
1. Run `prepare_custom_dataset.py` locally to generate the 3 `.npy` files
2. Upload these 3 files to your Google Drive:
   - `queries.npy`
   - `products.npy`
   - `labels.npy`
3. Set **Runtime > Change runtime type > T4 GPU**

The trained model (`pytorch_model.bin`) will be saved back to your Google Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths
Edit the paths below to match where you uploaded your files in Google Drive.

In [ ]:
import os

# --- EDIT THESE PATHS ---
# Folder in your Google Drive where you uploaded queries.npy, products.npy, labels.npy
NPY_DIR = '/content/drive/MyDrive/esci_prepared'

# Folder where checkpoints and trained model will be saved
MODEL_OUTPUT_DIR = '/content/drive/MyDrive/esci_trained_model'
# ------------------------

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

OUT_QUERIES  = os.path.join(NPY_DIR, 'queries.npy')
OUT_PRODUCTS = os.path.join(NPY_DIR, 'products.npy')
OUT_LABELS   = os.path.join(NPY_DIR, 'labels.npy')

# Verify files exist
for f in [OUT_QUERIES, OUT_PRODUCTS, OUT_LABELS]:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f)/1e9:.2f} GB' if exists else 'MISSING'
    print(f'  {os.path.basename(f)}: {size}')

## 3. Install dependencies

In [ ]:
!pip install transformers==4.40.0 scikit-learn seaborn -q

## 4. Imports & model definition

In [ ]:
import numpy as np
import random
import time
import torch
import torch.nn as nn
from collections import Counter
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from torch.nn.utils import clip_grad_norm_
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
ESCI_LABEL_NAMES = ['Exact', 'Substitute', 'Complement', 'Irrelevant']

class QueryProductClassifier(nn.Module):
    def __init__(self, size_pretrained=768, dense_hidden_dim=256, num_dense_layers=2, num_labels=4, dropout_rate=0.1):
        super().__init__()
        self.num_labels = 1 if num_labels <= 2 else num_labels
        prev_dim = size_pretrained * 2
        self.dropout_embedding = nn.Dropout(dropout_rate)
        fc_layers = []
        for _ in range(num_dense_layers):
            fc_layers += [nn.Linear(prev_dim, dense_hidden_dim), nn.BatchNorm1d(dense_hidden_dim), nn.ReLU(), nn.Dropout(dropout_rate)]
            prev_dim = dense_hidden_dim
        fc_layers.append(nn.Linear(prev_dim, self.num_labels))
        self.fc = nn.Sequential(*fc_layers)

    def forward(self, q, p):
        return self.fc(self.dropout_embedding(torch.cat((q, p), 1))).squeeze(-1)


def generate_dataset(q, p, y):
    return TensorDataset(torch.tensor(q).float(), torch.tensor(p).float(), torch.tensor(y))


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_class_weights(labels, num_classes):
    counts = Counter(labels.tolist())
    total  = len(labels)
    w = [total / (num_classes * counts.get(i, 1)) for i in range(num_classes)]
    min_w = min(w)
    return [x / min_w for x in w]

print('Classes defined.')

## 5. Load .npy files

In [ ]:
query_array  = np.load(OUT_QUERIES)
asin_array   = np.load(OUT_PRODUCTS)
labels_array = np.load(OUT_LABELS)

print(f'queries  : {query_array.shape}')
print(f'products : {asin_array.shape}')
print(f'labels   : {labels_array.shape}')
print(f'Label distribution: {Counter(labels_array.tolist())}')

## 6. Train/dev split

In [ ]:
NUM_DEV_EXAMPLES = 5505
RANDOM_STATE     = 42

dev_size = NUM_DEV_EXAMPLES / len(labels_array)
ids_train, ids_dev = train_test_split(range(len(labels_array)), test_size=dev_size, random_state=RANDOM_STATE)

train_dataset = generate_dataset(query_array[ids_train], asin_array[ids_train], labels_array[ids_train])
dev_dataset   = generate_dataset(query_array[ids_dev],   asin_array[ids_dev],   labels_array[ids_dev])

print(f'Train: {len(train_dataset):,}  Dev: {len(dev_dataset):,}')

## 7. Training config

In [ ]:
# --- Tune these if needed ---
BATCH_SIZE        = 512   # larger = faster on GPU
NUM_EPOCHS        = 5
LR                = 5e-5
WEIGHT_DECAY      = 0.01
VALIDATION_STEPS  = 250
USE_CLASS_WEIGHTS = True
# ----------------------------

set_seed(RANDOM_STATE)

model = QueryProductClassifier(num_labels=4).to(device)

train_loader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset),   batch_size=BATCH_SIZE)
val_loader   = DataLoader(dev_dataset,   sampler=SequentialSampler(dev_dataset), batch_size=BATCH_SIZE)

total_steps = len(train_loader) * NUM_EPOCHS
optimizer   = torch.optim.Adam(model.parameters(), lr=LR, eps=1e-8, weight_decay=WEIGHT_DECAY)
scheduler   = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

if USE_CLASS_WEIGHTS:
    cw = compute_class_weights(labels_array[ids_train], 4)
    print(f'Class weights: {[round(w,3) for w in cw]}')
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device))
else:
    criterion = nn.CrossEntropyLoss()

print(f'Model params : {sum(p.numel() for p in model.parameters()):,}')
print(f'Total steps  : {total_steps:,}')

## 8. Train

In [ ]:
best_metric  = 0.0
global_step  = 0
start_epoch  = 0
val_m_arr    = np.empty(len(val_loader))
val_l_arr    = np.empty(len(val_loader))
step_start   = time.time()

# Resume from checkpoint if exists
existing_ckpts = sorted([f for f in os.listdir(MODEL_OUTPUT_DIR)
                         if f.startswith('checkpoint_epoch_') and f.endswith('.pt')])
if existing_ckpts:
    ckpt_path = os.path.join(MODEL_OUTPUT_DIR, existing_ckpts[-1])
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch = ckpt['epoch'] + 1
    global_step = ckpt['global_step']
    best_metric = ckpt['best_metric_value']
    print(f'Resumed from {existing_ckpts[-1]} — epoch {start_epoch}, best acc {best_metric:.4f}')

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f'\n=== Epoch {epoch+1}/{NUM_EPOCHS} ===')

    for batch_idx, batch in enumerate(train_loader):
        model.train()
        labels = batch[2].to(device)
        optimizer.zero_grad()
        logits = model(batch[0].to(device), batch[1].to(device))
        loss   = criterion(logits.view(-1, 4), labels.view(-1))
        preds  = np.argmax(logits.detach().cpu().numpy(), axis=1)
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        global_step += 1

        if batch_idx % VALIDATION_STEPS == 0:
            elapsed  = time.time() - step_start
            s_per_it = elapsed / global_step if global_step > 0 else 0
            eta_min  = s_per_it * (total_steps - global_step) / 60
            t_acc    = accuracy_score(labels.detach().cpu().numpy(), preds)
            print(f'  Train  Ep{epoch+1} Step{global_step}/{total_steps} Loss:{loss:.3f} Acc:{t_acc:.3f} ETA:{eta_min:.1f}min')

            model.eval()
            for vi, vb in enumerate(val_loader):
                vl = vb[2].to(device)
                with torch.no_grad():
                    vlogits = model(vb[0].to(device), vb[1].to(device))
                val_m_arr[vi] = accuracy_score(vl.cpu().numpy(), np.argmax(vlogits.cpu().numpy(), axis=1))
                val_l_arr[vi] = criterion(vlogits.view(-1, 4), vl.view(-1)).item()
            val_acc = np.mean(val_m_arr)
            print(f'  Val    Ep{epoch+1} Step{global_step}/{total_steps} Loss:{np.mean(val_l_arr):.3f} Acc:{val_acc:.3f}')

            if val_acc > best_metric:
                best_metric = val_acc
                torch.save(model.state_dict(), os.path.join(MODEL_OUTPUT_DIR, 'pytorch_model.bin'))
                print(f'  >> Best model saved (acc={best_metric:.4f})')

    # Save epoch checkpoint
    new_ckpt = os.path.join(MODEL_OUTPUT_DIR, f'checkpoint_epoch_{epoch+1}.pt')
    torch.save({
        'epoch': epoch, 'global_step': global_step,
        'best_metric_value': best_metric,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
    }, new_ckpt)
    if epoch > start_epoch:
        prev = os.path.join(MODEL_OUTPUT_DIR, f'checkpoint_epoch_{epoch}.pt')
        if os.path.exists(prev): os.remove(prev)
    print(f'Epoch {epoch+1} checkpoint saved.')

print(f'\nDone. Best validation accuracy: {best_metric:.4f}')

## 9. Final Evaluation

In [ ]:
best_model = QueryProductClassifier(num_labels=4)
best_model.load_state_dict(torch.load(os.path.join(MODEL_OUTPUT_DIR, 'pytorch_model.bin'), map_location=device))
best_model.to(device).eval()

all_true, all_pred = [], []
for batch in DataLoader(dev_dataset, sampler=SequentialSampler(dev_dataset), batch_size=BATCH_SIZE):
    with torch.no_grad():
        logits = best_model(batch[0].to(device), batch[1].to(device))
    all_true.extend(batch[2].numpy())
    all_pred.extend(np.argmax(logits.cpu().numpy(), axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(classification_report(all_true, all_pred, target_names=ESCI_LABEL_NAMES))

cm = confusion_matrix(all_true, all_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=ESCI_LABEL_NAMES, yticklabels=ESCI_LABEL_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=ESCI_LABEL_NAMES, yticklabels=ESCI_LABEL_NAMES, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalised)')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print(f'Confusion matrix saved to Drive.')

## 10. Download trained model

In [ ]:
from google.colab import files
files.download(os.path.join(MODEL_OUTPUT_DIR, 'pytorch_model.bin'))
print('Download started. Place the file at:')
print('RetailTalk/backend/models/esci_classifier/pytorch_model.bin')